# AdaBoost Classifier — Geometric Intuition

## 1. Foundational Concepts

### A. Weak Learners

- A weak learner is a model that performs only slightly better than random guessing.
- In binary classification:
  - Random = 50% accuracy
  - Weak learner = ~51% to 53% accuracy

### Key Idea:
AdaBoost does not rely on one strong model. Instead, it builds a strong model by combining many weak learners sequentially.

---

### B. Decision Stumps

- A decision stump is a Decision Tree with:
$$
\text{max\_depth} = 1
$$

#### Properties:
- Uses only one feature for splitting
- Creates a single axis-aligned decision boundary
- Very simple → high bias → weak individually

#### Geometric Intuition:
- The decision boundary is a straight line (2D) or hyperplane (higher dimensions)
- Always parallel to feature axes

---

### C. Labeling Scheme (+1 / -1)

AdaBoost uses:

- Positive class → $+1$
- Negative class → $-1$

#### Why?

This simplifies:
- Loss function (exponential loss)
- Final prediction rule (sign function)

---

## 2. Stage-Wise Additive Modeling

AdaBoost is built on:

### Sequential Learning Process

- Models are trained one after another
- Each model corrects errors of previous ones

#### Contrast:

| Algorithm       | Strategy        |
|----------------|----------------|
| Random Forest  | Parallel voting |
| AdaBoost       | Sequential correction |

---

## 3. Step-by-Step Geometric Intuition

---

## Stage 1: Initial Model

1. All data points start with equal weight.
2. Train first weak learner:
$
h_1(x)
$
3. It creates the first decision boundary (axis-aligned split).
4. Some points are misclassified due to simplicity.

### Example:
If class separation is circular but stump is vertical split → many errors occur.

---

## Stage 2: Adaptive Weight Update

1. Identify misclassified points from $h_1(x)$
2. Increase their weights

### Intuition:
- Misclassified points become "more important"
- They visually "expand" in influence

3. Train next weak learner:
$
h_2(x)
$

### Effect:
- New model focuses more on previously misclassified points
- May sacrifice correctness on previously correct points

---

## Stage 3: Repetition (Boosting Process)

Repeat for $M$ models:$
h_1(x), h_2(x), ..., h_M(x)
$

At each step:
- Misclassified points get higher weight
- Model adapts to hardest cases progressively

---

## 4. Final Model: Weighted Voting

Unlike Random Forest (equal voting), AdaBoost uses weighted voting.

### Final Prediction:

$
F(x) = \text{sign} \left( \sum_{m=1}^{M} \alpha_m h_m(x) \right)
$

Where:
- $h_m(x)$ = weak learner output
- $\alpha_m$ = importance of learner
- sign() decides final class

---

### Meaning of $\alpha$

- High accuracy model → large $\alpha$
- Weak model → small $\alpha$

#### Intuition:
Better models have more "say" in final decision.

---

## 5. Practical Example

### Given:

| Model | Weight ($\alpha$) | Prediction |
|------|------------------|-----------|
| $h_1$ | 2.0 | -1 |
| $h_2$ | 4.0 | +1 |
| $h_3$ | 1.0 | -1 |

---

### Step-by-step:

$
F(x) = \text{sign} \left( 2(-1) + 4(1) + 1(-1) \right)
$

$
F(x) = \text{sign}(-2 + 4 - 1)
$

$
F(x) = \text{sign}(1) = +1
$

---

### Insight:

Even though:
- 2 models vote -1
- Only 1 model votes +1

The stronger model dominates due to higher $\alpha$.

---

## 6. Geometric Intuition of Boosting

### Key Idea:

Each stump creates a simple axis-aligned boundary.

After many iterations:

- Boundaries stack together
- Weighted combination forms a complex shape

### Result:

A highly non-linear decision boundary that:
- Wraps around clusters
- Focuses on difficult points
- Reduces bias progressively

---

## 7. Final Mental Model (Very Important)

Think of AdaBoost as:

> A team of weak classifiers where each new member focuses only on past mistakes, and smarter members get more voting power.

### Flow:

1. Equal importance → first model
2. Mistakes identified
3. Hard points amplified
4. Next model focuses on them
5. Repeat
6. Weighted voting gives final decision

---

## 8. One-Line Summary

AdaBoost converts many weak axis-aligned decision stumps into a powerful nonlinear classifier by sequentially reweighting errors and combining learners using a weighted majority vote.

# AdaBoost Classifier — Step-by-Step Mathematical Mechanics

This notebook explains the full mathematical pipeline of AdaBoost:
- Weight initialization
- Weak learner training
- Error computation
- Model weight ($\alpha$) calculation
- Sample weight update
- Dataset resampling (up-sampling intuition)

---

## 1. Initial State: Sample Weight Initialization

Assume a dataset with:

- $N = 5$ samples
- Binary labels: $y \in \{-1, +1\}$

At initialization, all samples have equal importance.

### Initial weight per sample:

$
w_i = \frac{1}{N} = \frac{1}{5} = 0.20
$

| Row ID | $X_1$ | $X_2$ | $y$ | Initial Weight $w_i$ |
|--------|------|------|-----|----------------|
| 1 | ... | ... | $+1$ | 0.20 |
| 2 | ... | ... | $-1$ | 0.20 |
| 3 | ... | ... | $+1$ | 0.20 |
| 4 | ... | ... | $+1$ | 0.20 |
| 5 | ... | ... | $-1$ | 0.20 |

---

## 2. Training First Weak Learner $h_1(x)$

A decision stump is trained using weighted data.

- It evaluates all possible splits across features.
- Selects split minimizing weighted classification error.

### Example outcome:

Misclassified samples: **Row 2 and Row 5**

| Row ID | True $y$ | Prediction $\hat{y}$ | Status |
|--------|----------|----------------------|--------|
| 1 | +1 | +1 | Correct |
| 2 | -1 | +1 | Misclassified |
| 3 | +1 | +1 | Correct |
| 4 | +1 | +1 | Correct |
| 5 | -1 | +1 | Misclassified |

---

## 3. Weighted Error Calculation

AdaBoost uses **weighted error**, not simple misclassification count.

$
\epsilon_1 = \sum_{i \in \text{misclassified}} w_i
$

$
\epsilon_1 = w_2 + w_5 = 0.20 + 0.20 = 0.40
$

---

## 4. Model Weight ($\alpha_1$) Calculation

### Core formula:

$
\alpha_m = \frac{1}{2} \ln \left( \frac{1 - \epsilon_m}{\epsilon_m} \right)
$

---

### Interpretation:

- If $\epsilon \to 0$ → $\alpha \to \infty$ (perfect model)
- If $\epsilon = 0.5$ → $\alpha = 0$ (random guessing)
- If $\epsilon > 0.5$ → negative $\alpha$ (model is worse than random)

---

### Compute $\alpha_1$:

$
\alpha_1 = \frac{1}{2} \ln \left( \frac{1 - 0.40}{0.40} \right)
$

$
\alpha_1 = \frac{1}{2} \ln(1.5)
$

$
\alpha_1 \approx \frac{1}{2} \times 0.405 = 0.202
$

---

## 5. Sample Weight Update Rule

AdaBoost increases weights of misclassified samples.

### Update equations:

### If correctly classified:
$
w_i^{new} = w_i \cdot e^{-\alpha_1}
$

### If misclassified:
$
w_i^{new} = w_i \cdot e^{\alpha_1}$

---

## 6. Weight Update Computation

### Given:
$
\alpha_1 = 0.202
$

### Misclassified samples (Row 2, 5):

$
w = 0.20 \cdot e^{0.202}
$

$
w \approx 0.20 \cdot 1.224 = 0.245
$

---

### Correct samples (Row 1, 3, 4):

$
w = 0.20 \cdot e^{-0.202}
$

$
w \approx 0.20 \cdot 0.817 = 0.163
$

---

## 7. Normalization of Weights

Raw weights must sum to 1.

### Sum:

$
\sum w = 0.163 + 0.245 + 0.163 + 0.163 + 0.245 = 0.979
$

---

### Normalized weights:

$
w_i^{norm} = \frac{w_i}{0.979}
$

| Row | Status | Raw Weight | Normalized Weight |
|-----|--------|------------|------------------|
| 1 | Correct | 0.163 | 0.166 |
| 2 | Misclassified | 0.245 | 0.250 |
| 3 | Correct | 0.163 | 0.166 |
| 4 | Correct | 0.163 | 0.166 |
| 5 | Misclassified | 0.245 | 0.250 |

---

## 8. Dataset Reconstruction (Up-Sampling Intuition)

New dataset is generated by sampling using **cumulative probability distribution**.

---

### Cumulative Ranges:

| Row | Range |
|-----|------|
| 1 | [0.000, 0.166) |
| 2 | [0.166, 0.416) |
| 3 | [0.416, 0.582) |
| 4 | [0.582, 0.748) |
| 5 | [0.748, 1.000] |

---

### Key Insight:

- Misclassified rows (2 and 5) have higher probability mass (~0.250 each)
- Correct rows have lower probability (~0.166)

Thus:
- Misclassified samples are more likely to be repeated
- Dataset becomes biased toward hard examples

---

## 9. Final Step: Reset for Next Iteration

After resampling:

- New dataset is created
- Sample weights are reset to uniform
- Next weak learner $h_2(x)$ is trained

$
w_i = \frac{1}{N}
$

---

## 10. Core Intuition Summary

AdaBoost works as:

1. Train weak learner
2. Measure weighted error
3. Assign importance ($\alpha$)
4. Increase weight of misclassified points
5. Resample dataset
6. Repeat

---

## 11. Final Mental Model

AdaBoost is:

> A system that continuously forces the model to focus more on its previous mistakes, while giving stronger models more influence in the final decision.

---

## 12. One-Line Summary

AdaBoost is a sequential reweighting and resampling algorithm where each weak learner focuses more on previously misclassified samples, and all learners are combined using a weighted vote based on their performance.